In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.linear_model import LinearRegression

# Paths
raw_data_path = "/Users/khoale/Downloads/Alzheimer_Code/csvs/adni_mri_ucsf_merged.csv"
output_path = "/Users/khoale/Downloads/Alzheimer_Code/adni_mri_sustain_prepared_asymmetric.csv"

# Load the raw dataset
df = pd.read_csv(raw_data_path)
print(f"Loaded raw data: {df.shape[0]} subjects, {df.shape[1]} columns")

Loaded raw data: 577 subjects, 355 columns


In [2]:
# Map columns belonging to each of our 24 asymmetric biomarkers
biomarker_mapping = {
    # --- LEFT HEMISPHERE ---
    "L_Frontal": ["ST15CV", "ST25CV", "ST36CV", "ST39CV", "ST43CV", "ST45CV", "ST46CV", "ST47CV", "ST51CV", "ST55CV", "ST56CV"],
    "L_Temporal": ["ST13CV", "ST24CV", "ST26CV", "ST32CV", "ST40CV", "ST44CV", "ST58CV", "ST60CV", "ST62CV"],
    "L_Parietal": ["ST31CV", "ST49CV", "ST52CV", "ST57CV", "ST59CV"],
    "L_Occipital": ["ST23CV", "ST35CV", "ST38CV", "ST48CV"],
    "L_Cingulate": ["ST14CV", "ST34CV", "ST50CV", "ST54CV"],
    "L_Insula": ["ST129CV"],
    "L_Hippocampus": ["ST29SV"],
    "L_Amygdala": ["ST12SV"],
    "L_Caudate": ["ST16SV"],
    "L_Pallidum": ["ST42SV"],
    "L_Putamen": ["ST53SV"],
    "L_Accumbens": ["ST11SV"],

    # --- RIGHT HEMISPHERE ---
    "R_Frontal": ["ST74CV", "ST84CV", "ST95CV", "ST98CV", "ST102CV", "ST104CV", "ST105CV", "ST106CV", "ST110CV", "ST114CV", "ST115CV"],
    "R_Temporal": ["ST72CV", "ST83CV", "ST85CV", "ST91CV", "ST99CV", "ST103CV", "ST117CV", "ST119CV", "ST121CV"],
    "R_Parietal": ["ST90CV", "ST108CV", "ST111CV", "ST116CV", "ST118CV"],
    "R_Occipital": ["ST82CV", "ST94CV", "ST97CV", "ST107CV"],
    "R_Cingulate": ["ST73CV", "ST93CV", "ST109CV", "ST113CV"],
    "R_Insula": ["ST130CV"],
    "R_Hippocampus": ["ST88SV"],
    "R_Amygdala": ["ST71SV"],
    "R_Caudate": ["ST75SV"],
    "R_Pallidum": ["ST101SV"],
    "R_Putamen": ["ST112SV"],
    "R_Accumbens": ["ST70SV"]
}

In [3]:
# 1. Force all ROI columns and ICV (ST10CV) to numeric values
all_roi_cols = [col for cols in biomarker_mapping.values() for col in cols]
cols_to_convert = all_roi_cols + ["ST10CV"]

for col in cols_to_convert:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# 2. Drop rows where Intracranial Volume (ICV) is missing
df = df.dropna(subset=["ST10CV"]).copy()

# 3. Sum sub-regions for Left and Right separately
aggregated_df = pd.DataFrame()
aggregated_df["PTID"] = df["PTID"]
aggregated_df["Label"] = df["Label"]
aggregated_df["ICV"] = df["ST10CV"]

for region, cols in biomarker_mapping.items():
    aggregated_df[region] = df[cols].sum(axis=1, min_count=1)

print(f"Aggregated asymmetric data shape: {aggregated_df.shape}")
aggregated_df.head()

Aggregated asymmetric data shape: (577, 27)


,PTID,Label,ICV,L_Frontal,L_Temporal,L_Parietal,L_Occipital,L_Cingulate,L_Insula,L_Hippocampus,...,R_Parietal,R_Occipital,R_Cingulate,R_Insula,R_Hippocampus,R_Amygdala,R_Caudate,R_Pallidum,R_Putamen,R_Accumbens
0,003_S_1057,pMCI,1.352138e+06,72851.0,50525.0,42733.0,20813.0,9146.0,6286.0,3421.0,...,42838.0,18902.0,7182.0,5530.0,3270.5,1159.7,3268.1,1897.3,4703.5,418.7
1,003_S_1059,AD,1.520293e+06,57614.0,39750.0,39752.0,20244.0,7849.0,5941.0,2223.2,...,40367.0,21093.0,6507.0,6222.0,2668.8,877.1,3165.2,1482.6,2885.1,296.9
2,003_S_1257,AD,1.951182e+06,73453.0,47058.0,43438.0,22424.0,7604.0,6179.0,4644.7,...,48298.0,24454.0,6380.0,7278.0,4261.7,1700.3,5461.0,1741.9,3455.8,318.1
3,003_S_4119,CN,1.475697e+06,83368.0,55766.0,49947.0,21359.0,9471.0,8030.0,3682.4,...,53686.0,21678.0,9403.0,6945.0,3812.5,1713.2,3695.0,1801.5,4378.5,428.8
4,003_S_4152,AD,1.383684e+06,68657.0,39745.0,39933.0,26387.0,9903.0,6804.0,3267.5,...,43430.0,28127.0,9055.0,6660.0,3886.2,1419.3,2610.1,2338.7,4187.5,507.1


In [4]:
# Create a copy to store ICV-normalized volumes
normalized_df = aggregated_df.copy()

# Define the control (CN) cohort to fit the ICV regression line
cn_controls = aggregated_df[aggregated_df["Label"] == "CN"].copy()

print(f"Fitting ICV regression using {len(cn_controls)} CN controls...")

regions = list(biomarker_mapping.keys())

for region in regions:
    clean_cn = cn_controls.dropna(subset=[region, "ICV"])
    
    X_cn = clean_cn[["ICV"]].values
    y_cn = clean_cn[region].values
    
    # Fit linear model: Region = beta * ICV + alpha
    model = LinearRegression()
    model.fit(X_cn, y_cn)
    
    # Calculate residuals for ALL subjects: residual = actual - predicted
    X_all = aggregated_df[["ICV"]].values
    y_all = aggregated_df[region].values
    
    predicted_all = model.predict(X_all)
    residuals = y_all - predicted_all
    
    # Add back the mean volume of controls to restore volumetric scale
    mean_volume_cn = y_cn.mean()
    normalized_df[region] = residuals + mean_volume_cn

print("ICV correction complete.")

Fitting ICV regression using 186 CN controls...
ICV correction complete.


In [5]:
sustain_ready_df = pd.DataFrame()
sustain_ready_df["PTID"] = normalized_df["PTID"]
sustain_ready_df["Label"] = normalized_df["Label"]

# Fit z-score baseline using the ICV-normalized controls
cn_normalized = normalized_df[normalized_df["Label"] == "CN"]

for region in regions:
    mean_cn = cn_normalized[region].mean()
    std_cn = cn_normalized[region].std() + 1e-9
    
    # Inverted formula for atrophy: Z = (mean_healthy - actual) / std_healthy
    sustain_ready_df[region] = (mean_cn - normalized_df[region]) / std_cn

sustain_ready_df.head()

,PTID,Label,L_Frontal,L_Temporal,L_Parietal,L_Occipital,L_Cingulate,L_Insula,L_Hippocampus,L_Amygdala,...,R_Parietal,R_Occipital,R_Cingulate,R_Insula,R_Hippocampus,R_Amygdala,R_Caudate,R_Pallidum,R_Putamen,R_Accumbens
0,003_S_1057,pMCI,0.121356,-0.468780,0.959199,0.038303,-0.360036,0.217073,0.511956,1.340271,...,1.121592,1.279456,1.093153,0.988743,1.220659,1.581741,-0.035649,-0.361175,-1.001417,0.341345
1,003_S_1059,AD,3.093314,2.752515,2.209627,0.572984,1.340243,1.226664,3.979986,3.142252,...,2.258369,0.796580,2.266401,0.623351,3.000930,3.093743,0.548388,1.641775,2.735812,1.825023
2,003_S_1257,AD,2.246468,2.690647,2.940643,0.472656,2.868254,2.312848,-1.482346,-0.115701,...,2.102610,0.448979,3.612640,0.710564,-0.251585,-0.045871,-3.446410,1.637250,2.688977,1.833197
3,003_S_4119,CN,-1.042773,-1.235638,-0.175783,0.041514,-0.285076,-1.712937,0.077094,-0.784829,...,-0.767742,0.464210,-0.859209,-0.573413,0.075524,-0.757512,-0.684640,0.298737,-0.115765,0.297924
4,003_S_4152,AD,0.884820,2.221512,1.683129,-2.157432,-0.960267,-0.372974,0.976474,1.239349,...,1.107744,-2.291404,-0.758112,-0.507129,-0.261594,0.448323,1.444240,-2.002983,0.021432,-0.646556


In [6]:
# 1. Fill missing values (NaNs) with 0 (normal)
sustain_ready_df[regions] = sustain_ready_df[regions].fillna(0)

# 2. Clamp negative z-scores to 0
sustain_ready_df[regions] = sustain_ready_df[regions].clip(lower=0)

print("Preprocessing complete! Preview of final asymmetric z-scores:")
sustain_ready_df[regions].describe()

Preprocessing complete! Preview of final asymmetric z-scores:


,L_Frontal,L_Temporal,L_Parietal,L_Occipital,L_Cingulate,L_Insula,L_Hippocampus,L_Amygdala,L_Caudate,L_Pallidum,...,R_Parietal,R_Occipital,R_Cingulate,R_Insula,R_Hippocampus,R_Amygdala,R_Caudate,R_Pallidum,R_Putamen,R_Accumbens
count,577.000000,577.000000,577.000000,577.000000,577.000000,577.000000,577.000000,577.000000,577.000000,577.000000,...,577.000000,577.000000,577.000000,577.000000,577.000000,577.000000,577.000000,577.000000,577.000000,577.000000
mean,0.847103,1.281013,0.956025,0.591082,0.738373,0.731395,1.195355,1.080077,0.475456,0.512506,...,0.946966,0.644625,0.656930,0.747502,1.152307,0.975169,0.483736,0.491028,0.648590,0.687406
std,1.235173,1.577741,1.312125,0.807734,1.089511,1.084150,1.242347,1.086533,0.651128,0.826513,...,1.284415,0.879501,1.107633,1.208841,1.218561,1.068546,0.692548,0.779700,0.977805,0.764400
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.443781,0.918343,0.425832,0.211238,0.361818,0.418971,0.910153,0.867652,0.209644,0.138245,...,0.551100,0.302224,0.209757,0.356004,0.834696,0.643356,0.166559,0.205378,0.349857,0.417796
75%,1.232116,1.931630,1.552858,1.023123,1.107764,1.041303,1.994409,1.847931,0.771945,0.851915,...,1.398800,1.062868,0.880080,1.048936,1.915132,1.622650,0.767033,0.756258,0.935537,1.201800
max,11.805032,13.488634,11.514620,5.991918,9.573103,9.161613,8.890493,7.078721,5.670208,8.234888,...,11.730475,6.769774,9.487595,11.921110,7.970402,6.927289,6.409214,7.291370,10.586089,4.053256


In [7]:
# Save the prepared asymmetric dataset
sustain_ready_df.to_csv(output_path, index=False)
print(f"Asymmetric dataset exported to: {output_path}")

Asymmetric dataset exported to: /Users/khoale/Downloads/Alzheimer_Code/adni_mri_sustain_prepared_asymmetric.csv
